# The Rest of the Iceberg — exploring *Titanic Birth Records* with Quarb

Pandas is the perfect tool for table-shaped data. But a table
presupposes that somebody made the data table-shaped first. This
notebook does that below-the-waterline work on **Titanic Birth Records**: the
classic `titanic.csv` plus `titanic-birth-records.json`, a fictional but
historically careful reconstruction of birth records for the ship's
passengers, drawn from eight registration systems of 1912 — eight
different tree shapes in one file.

It accompanies the article
[*The rest of the iceberg*](https://quarb.org/articles/the-rest-of-the-iceberg.html).
Setup:

```
pip install "quarb[jupyter]"
```

and put `titanic.csv` and `titanic-birth-records.json` (from the
[Kaggle dataset](https://www.kaggle.com/datasets/deyanovich/titanic-birth-records) *Titanic Birth Records* or
[data.quarb.org](https://data.quarb.org/articles/titanic-birth-records-fixtures.tar.zst)) next to this
notebook.

In [1]:
%load_ext quarb.ipython
%quarb_mount titanic.csv titanic-birth-records.json

mounted: titanic+titanic-birth-records (children: /titanic, /titanic-birth-records)


## First contact

The JSON is a stranger. Ask it what it is — `/*` walks children,
`:::name` asks a node its name, `//` searches at any depth, and the
two pipes split the work: `|` streams one result at a time, `@|`
hands the whole stream to an aggregate.

In [2]:
%%quarb
/titanic-birth-records/archives/*:::name

gro
irish_parishes
kyrkobocker
actes_de_naissance
us_state_records
ottoman_nufus
matricula_baptizatorum
metricheskie_knigi

In [3]:
%%quarb
/titanic-birth-records//passenger_id:: @| count

1146

In [4]:
%%quarb
/titanic-birth-records//*:::name @| unique @| count

534

1,146 records; 534 distinct key names across eight archive
shapes. Which bureaucracy knew about how many passengers? Walk to
each archive, file its name in the register with a push `.a(...)`,
keep walking, group by the pushed name:

In [5]:
%%quarb
/titanic-birth-records/archives/* | .a(:::name) | //passenger_id @| group($.a) | count | .n | %.

a,n
gro,422
irish_parishes,124
kyrkobocker,166
actes_de_naissance,27
us_state_records,286
ottoman_nufus,80
matricula_baptizatorum,34
metricheskie_knigi,7


## The table meets the tree

The famous half: 891 rows, and the famous hole — 177 rows with no
`Age`, the ones every tutorial fills with the median. The join
operator `<=>` correlates two expressions (the first expression
drives, `$$` names its row inside the join condition, `$*1` the
joined binding after); the outer form `<=>?` keeps unmatched rows
with a null witness, so an anti-join is a join plus a null
test.

In [6]:
%%quarb
/titanic/row[not ::Age] @| count

177

In [7]:
%%quarb
/titanic-birth-records//*[::passenger_id <= 891] @| count

773

In [8]:
%%quarb
/titanic/row <=>? /titanic-birth-records//*[::passenger_id = $$::PassengerId]
 | [not $*1::passenger_id] | ::Name @| [..6]

Vestrom, Miss. Hulda Amanda Adolfina
Hewlett, Mrs. (Mary D Kingcome) 
Vander Planke, Mrs. Julius (Emelia Maria Vandemoortele)
Todoroff, Mr. Lalio
Uruchurtu, Don. Manuel E
Vander Planke, Miss. Augusta Maria

In [9]:
%%quarb
/titanic-birth-records//*[::passenger_id > 891] @| count

373

773 of our rows have a record. The 118 without one are mostly
passengers born outside the eight covered systems — Belgians,
Bulgarians, a Mexican diplomat. And 373 records cite passengers
*beyond* id 891: the registers know the whole ship; the table only
ever held the training set.

## One year, eight spellings

A birth year is spelled `quarter: "1890 Q1"` in London,
`date: "3 October 1892"` in Mayo, `född: "1871-11-26"` in Sweden,
`year_rumi: 1301` in Ottoman Batroun, a Julian date in Kovno. One
conditional ladder normalizes all eight — each branch tests for a
key only one shape carries ("has this key" is how you say "is this
shape" when shapes have no names). Join the 177 age-less rows to
their records and read the answer:

In [10]:
%%quarb
def &gregorian: | tp("%Y-%m-%d") | ($_ + 12d);
/titanic/row[not ::Age] <=> /titanic-birth-records//*[::passenger_id = $$::PassengerId]
 | .born(( $*1::year_rumi      ? $*1::year_rumi + 584
         : $*1::quarter        ? ($*1::quarter | s/ Q.*$//) * 1
         : $*1::date           ? ($*1::date | s/^.*\s//) * 1
         : $*1::född           ? ($*1::född | s/-.*$//) * 1
         : $*1::né_le          ? ($*1::né_le | s/-.*$//) * 1
         : $*1::date_of_birth  ? ($*1::date_of_birth | s/-.*$//) * 1
         : $*1::natus          ? ($*1::natus | s/-.*$//) * 1
         : $*1::nata           ? ($*1::nata | s/-.*$//) * 1
         : (($*1::data_rozhdeniya | &gregorian) | s/-.*$//) * 1 ))
 | rec(::Name, "born", $.born, "age_in_1912", 1912 - $.born) @| [..10]

Name,born,age_in_1912
"Moran, Mr. James",1889,23
"Williams, Mr. Charles Eugene",1875,37
"Masselmani, Mrs. Fatima",1877,35
"Emir, Mr. Farred Chehab",1881,31
"O'Dwyer, Miss. Ellen ""Nellie""",1894,18
"Spencer, Mrs. William Augustus (Marie Eugenie)",1888,24
"Glynn, Miss. Mary Agatha",1895,17
"Mamee, Mr. Hanna",1885,27
"Rogers, Mr. William John",1871,41
"Lennon, Mr. Denis",1872,40


In [11]:
%%quarb
/titanic/row[not ::Age] <=> /titanic-birth-records//*[::passenger_id = $$::PassengerId] @| count

157

**157 of the 177 missing ages were never missing** — they were
in the parish books, waiting for the data-engineering pass instead
of the median.

## The passengers who lied

For rows that *do* carry an age, compare it with the records
(people lied to shipping lines; registers did not). One join, one
pushed subtraction, one filter:

In [12]:
%%quarb
def &gregorian: | tp("%Y-%m-%d") | ($_ + 12d);
/titanic/row[::Age] <=> /titanic-birth-records//*[::passenger_id = $$::PassengerId]
 | .born(( $*1::year_rumi      ? $*1::year_rumi + 584
         : $*1::quarter        ? ($*1::quarter | s/ Q.*$//) * 1
         : $*1::date           ? ($*1::date | s/^.*\s//) * 1
         : $*1::född           ? ($*1::född | s/-.*$//) * 1
         : $*1::né_le          ? ($*1::né_le | s/-.*$//) * 1
         : $*1::date_of_birth  ? ($*1::date_of_birth | s/-.*$//) * 1
         : $*1::natus          ? ($*1::natus | s/-.*$//) * 1
         : $*1::nata           ? ($*1::nata | s/-.*$//) * 1
         : (($*1::data_rozhdeniya | &gregorian) | s/-.*$//) * 1 ))
 | .gap(::Age - (1912 - $.born))
 | [abs($.gap) >= 2]
 | rec(::Name, "claimed", ::Age * 1, "records", 1912 - $.born)

Name,claimed,records
"Heikkinen, Miss. Laina",26,24
"Allen, Mr. William Henry",35,31
"Saundercock, Mr. William Henry",20,15
"Hickman, Mr. Stanley George",21,19
"Hakkarainen, Mrs. Pekka Pietari (Elin Matilda Dolck)",24,22
"Johnson, Miss. Eleanor Ileen",1,5
"Youseff, Mr. Gerious",45.5,42
"Givard, Mr. Hans Kristensen",30,25
"Leyson, Mr. Robert William Norman",24,19
"Maenpaa, Mr. Matti Alexanteri",22,17


## The sisters nobody's ticket reveals

A birth record names *parents*, and two records that agree on both
parents are siblings whatever the manifest says. Self-join the
Swedish church books, then ask the manifest who each pair boarded
as — the final predicate sees all three bound contexts:

In [13]:
%%quarb
/titanic-birth-records//födda/*
 <=> /titanic/row[::PassengerId = $$::passenger_id]
 <=> /titanic-birth-records//födda/*[::fader = $$::fader
       && ::moder = $$::moder && ::passenger_id > $$::passenger_id]
 | [$*1::Name *= "Mrs."]
 | rec("aboard_as", $*1::Name, "sibling", $*2::namn)

aboard_as,sibling
"Asplund, Mrs. Carl Oscar (Selma Augusta Emilia Johansson)",Gustaf Joel Johansson
"Ahlin, Mrs. Johan (Johanna Persdotter Larsson)",Bengt Edvin Larsson
"Backstrom, Mrs. Karl Alfred (Maria Mathilda Gustafsson)",Anders Vilhelm Gustafsson


Three married women, each traveling under a husband's name,
each with a brother aboard under the name she was born with. The
joining key — who your parents were — exists only below the
waterline.

## The names they left behind

Emigrants Westernized their names; the registers kept the
originals. Keep the records whose recorded surname does not appear
in the manifest name:

In [14]:
%%quarb
/titanic/row <=> /titanic-birth-records//*[::passenger_id = $$::PassengerId]
 | [$*1::name] | .sur(($*1::name | s/^.*\s//))
 | [not (::Name *= $.sur)]
 | rec("manifest", ::Name, "record", $*1::name, "village", $*1::village)

manifest,record,village
"Novel, Mr. Mansouer",Mansouer Nawfal,Hardin
"Peter, Miss. Anna",Anna Boutros,Shweir
"Thomas, Master. Assad Alexander",Assad Tuma,Tula


In [15]:
%%quarb
/titanic/row <=> /titanic-birth-records//*[::passenger_id = $$::PassengerId]
 | [$*1::familiya] | [not (::Name *= $*1::familiya)]
 | rec("manifest", ::Name,
       "record", "${$*1::imya} ${$*1::otchestvo} ${$*1::familiya}")

manifest,record
"Abelson, Mr. Samuel",Shmuel Gershovich Abelzon
"Paulner, Mr. Uscher",Usher Shmuelovich Polner


## The tip, at last — over to pandas

Now the data deserves a table: every passenger, the age the
manifest claims or the records recover, and the provenance of each
value. The same engine is an ordinary library, so the hand-off is
one call — `values()` returns plain dicts.

In [16]:
import pandas as pd
import quarb

root = quarb.mount(["titanic.csv", "titanic-birth-records.json"])
df = pd.DataFrame(root.values(r'''
def &gregorian: | tp("%Y-%m-%d") | ($_ + 12d);
/titanic/row <=>? /titanic-birth-records//*[::passenger_id = $$::PassengerId]
 | .born(( $*1::year_rumi      ? $*1::year_rumi + 584
         : $*1::quarter        ? ($*1::quarter | s/ Q.*$//) * 1
         : $*1::date           ? ($*1::date | s/^.*\s//) * 1
         : $*1::född           ? ($*1::född | s/-.*$//) * 1
         : $*1::né_le          ? ($*1::né_le | s/-.*$//) * 1
         : $*1::date_of_birth  ? ($*1::date_of_birth | s/-.*$//) * 1
         : $*1::natus          ? ($*1::natus | s/-.*$//) * 1
         : $*1::nata           ? ($*1::nata | s/-.*$//) * 1
         : (($*1::data_rozhdeniya | &gregorian) | s/-.*$//) * 1 ))
 | rec("id", ::PassengerId * 1, ::Name, ::Sex, "pclass", ::Pclass * 1,
       "survived", ::Survived * 1,
       "age", (( ::Age ? ::Age * 1 : 1912 - $.born )),
       "age_source", (( ::Age ? "manifest" : $.born ? "records" : "unknown" )))
'''))
df.shape

(891, 7)

In [17]:
df["age_source"].value_counts()

age_source
manifest    714
records     157
unknown      20
Name: count, dtype: int64

In [18]:
df.groupby("age_source")["survived"].mean().round(3)

age_source
manifest    0.406
records     0.318
unknown     0.100
Name: survived, dtype: float64

From 177 unknown ages to 20 — and a finding the median would
have buried: the passengers whose ages had to be *recovered*
survived at 32%, against 41% for the rows the manifest filled in.
The missingness was never random — it tracks third class, and
imputing the median quietly erases that structure.

Pandas is now rightfully in charge. The rest of the iceberg —
eight archives, five joins, one conditional ladder — is what it
took to earn this table.